#### Baseline Forecast Model — Departures (Sprint 6)
**Overview**
This notebook implements a baseline forecasting approach to predict hourly bike departures for downtown stations.
The baseline model uses simple lag-based predictors (1 hour, 24 hours, and 168 hours) to establish a reference performance benchmark for more advanced machine learning models.

**Scope**
- Target variable: **departures**
- Geographic scope: **downtown stations only**
- Temporal granularity: **station-hour level**
- Dataset: precomputed feature table (`downtown_dep_features_v1`)

**Purpose**
This baseline serves as a benchmark for evaluating the performance of:
- Linear Regression
- Random Forest
- XGBoost
- LightGBM

It provides a simple, interpretable reference before introducing more complex models.

## Baseline Methodology

The baseline model uses historical lag values as predictions:

- **Lag 1**: value from the previous hour
- **Lag 24**: value from the same hour on the previous day
- **Lag 168**: value from the same hour one week before

These predictors capture short-term and seasonal temporal patterns without using machine learning models.

Performance is evaluated using:
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)

A rolling evaluation strategy is applied using a fixed lookback window.

In [0]:
# ============================================================
# Baseline Forecast Model — Departures
# Progressive monthly evaluation using lag-based naive forecast
# ============================================================

from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# 0) Paths
# ------------------------------------------------------------
FEATURES_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/features/downtown_dep_features_v1"
EVAL_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/downtown_dep_baseline_progressive_v2"

TRAIN_LOOKBACK_DAYS = 90
TARGET = "target_departures"

# ------------------------------------------------------------
# 1) Load feature dataset
# ------------------------------------------------------------
df_feat = spark.read.parquet(FEATURES_DIR)

loaded_rows = df_feat.count()
print("Loaded feature rows:", f"{loaded_rows:,}")

# ------------------------------------------------------------
# 2) Available months
# ------------------------------------------------------------
months_rows = (
    df_feat.select("year", "month")
    .distinct()
    .orderBy("year", "month")
    .collect()
)

months_list = [(int(r["year"]), int(r["month"])) for r in months_rows]

# ------------------------------------------------------------
# 3) Month cache
# ------------------------------------------------------------
month_pd_cache = {}

def load_month_pd(y: int, m: int) -> pd.DataFrame:
    key = (y, m)
    if key in month_pd_cache:
        return month_pd_cache[key]

    sdf = (
        df_feat
        .filter((F.col("year") == y) & (F.col("month") == m))
        .select(
            "station_id", "year", "month", "day", "hour",
            "dow_num", "is_weekend",
            "lag1_dep", "lag24_dep", "lag168_dep",
            TARGET
        )
    )

    pdf = sdf.toPandas()

    if len(pdf) > 0:
        pdf["date"] = pd.to_datetime(pdf[["year", "month", "day"]])

    month_pd_cache[key] = pdf
    return pdf

def prev_months_in_lookback(test_start: pd.Timestamp, lookback_days: int):
    start = (test_start - pd.Timedelta(days=lookback_days)).to_period("M")
    end = (test_start - pd.Timedelta(days=1)).to_period("M")
    periods = pd.period_range(start, end, freq="M")
    return [(int(p.year), int(p.month)) for p in periods]

# ------------------------------------------------------------
# 4) Progressive evaluation loop
# ------------------------------------------------------------
results = []

for (y, m) in months_list:
    test_start = pd.Timestamp(year=y, month=m, day=1)
    test_end = test_start + pd.offsets.MonthEnd(0)

    train_end = test_start - pd.Timedelta(days=1)
    train_start = train_end - pd.Timedelta(days=TRAIN_LOOKBACK_DAYS)

    test_pd = load_month_pd(y, m)
    if test_pd.empty:
        continue

    train_parts = []
    for (yy, mm) in prev_months_in_lookback(test_start, TRAIN_LOOKBACK_DAYS):
        part = load_month_pd(yy, mm)
        if not part.empty:
            train_parts.append(part)

    if not train_parts:
        continue

    train_pd_all = pd.concat(train_parts, ignore_index=True)

    train_pd = train_pd_all[(train_pd_all["date"] >= train_start) & (train_pd_all["date"] <= train_end)]
    test_pd_f = test_pd[(test_pd["date"] >= test_start) & (test_pd["date"] <= test_end)]

    if train_pd.empty or test_pd_f.empty:
        continue

    y_true = test_pd_f[TARGET].astype(np.float32).values

    # Baseline 1: previous hour
    pred_lag1 = test_pd_f["lag1_dep"].astype(np.float32).values
    mae_lag1 = float(mean_absolute_error(y_true, pred_lag1))
    rmse_lag1 = float(np.sqrt(mean_squared_error(y_true, pred_lag1)))

    # Baseline 2: previous day same hour
    pred_lag24 = test_pd_f["lag24_dep"].astype(np.float32).values
    mae_lag24 = float(mean_absolute_error(y_true, pred_lag24))
    rmse_lag24 = float(np.sqrt(mean_squared_error(y_true, pred_lag24)))

    # Baseline 3: previous week same hour
    pred_lag168 = test_pd_f["lag168_dep"].astype(np.float32).values
    mae_lag168 = float(mean_absolute_error(y_true, pred_lag168))
    rmse_lag168 = float(np.sqrt(mean_squared_error(y_true, pred_lag168)))

    results.append({
        "year": y,
        "month": m,
        "rows_test": int(len(y_true)),
        "baseline_name": "lag1_naive",
        "baseline_mae": mae_lag1,
        "baseline_rmse": rmse_lag1,
        "lag24_mae": mae_lag24,
        "lag24_rmse": rmse_lag24,
        "lag168_mae": mae_lag168,
        "lag168_rmse": rmse_lag168
    })

# ------------------------------------------------------------
# 5) Results
# ------------------------------------------------------------
results_pd = pd.DataFrame(results).sort_values(["year", "month"])
display(results_pd)

# ------------------------------------------------------------
# 6) Global summary
# ------------------------------------------------------------
summary = pd.DataFrame({
    "Metric": [
        "Avg MAE (lag1)",
        "Avg RMSE (lag1)",
        "Avg MAE (lag24)",
        "Avg RMSE (lag24)",
        "Avg MAE (lag168)",
        "Avg RMSE (lag168)"
    ],
    "Value": [
        results_pd["baseline_mae"].mean(),
        results_pd["baseline_rmse"].mean(),
        results_pd["lag24_mae"].mean(),
        results_pd["lag24_rmse"].mean(),
        results_pd["lag168_mae"].mean(),
        results_pd["lag168_rmse"].mean()
    ]
})

display(summary)

print("Number of stations:", df_feat.select("station_id").distinct().count())
# ------------------------------------------------------------
# 7) Save outputs
# ------------------------------------------------------------
spark.createDataFrame(results_pd).write.mode("overwrite").parquet(EVAL_DIR)
print("Saved baseline evaluation to:", EVAL_DIR)

The baseline forecasting model establishes a simple reference point for evaluating more advanced predictive models. Three naive temporal benchmarks were evaluated: using the previous hour (lag1), the same hour from the previous day (lag24), and the same hour from the previous week (lag168).

Results show that the lag1 baseline consistently achieves the lowest average prediction error, indicating that short-term temporal continuity is the strongest signal in station-level bike departures. This suggests that demand tends to evolve gradually from one hour to the next.

These results establish a practical benchmark: any predictive model developed later (e.g., Linear Regression, Random Forest, XGBoost, or LightGBM) must outperform this naive baseline to demonstrate meaningful predictive capability.